# 16. Гибридная модель (Kaggle)

Гибрид объединяет три базовые модели — `lgbm_final`, `gnn_od_in`, `chronos2` — и учитывает их недавние ошибки
на станции, у соседей и по городу (`src/train_hybrid.py`).

Чтобы мета-модель училась на честных прогнозах, базовые модели заново обучаются до 30.09.2025 и прогнозируют
период stack (10.2025–03.2026, каждая 2-я точка). val и test при этом не трогаются.

| шаг | что | время |
|---|---|---|
| 1 | прогнозы базовых моделей на stack: Chronos-2 (GPU, в фоне) и LightGBM (CPU) параллельно, затем графовая сеть | ~1,5–2 ч |
| 2 | гибрид: 7 вариантов, val и test | ~1–1,5 ч |
| 3 | оценка, пары моделей, архив | ~15 мин |

**Input:** датасет `vkr-transit-data`, вывод ноутбуков `lgbm-final`, `gnn`, `chronos2` (Your Work → Notebooks).
Accelerator — **GPU T4 x1**, Internet — On. Результат — `results_csv.zip` в Output.

In [ ]:
!pip install -q chronos-forecasting==2.3.2

In [ ]:
import os, glob, shutil, subprocess, time
REPO = "/kaggle/working/vkr-transit"
if not os.path.exists(REPO):
    subprocess.run(["git", "clone", "--depth", "1", "https://github.com/Kinggl3/vkr-transit", REPO], check=True)
print(subprocess.run(["git", "-C", REPO, "log", "--oneline", "-1"], capture_output=True, text=True).stdout)
lab = [os.path.dirname(p) for p in glob.glob("/kaggle/input/**/ridership_wide.parquet", recursive=True)]
assert lab, "Не найден ridership_wide.parquet: подключите датасет vkr-transit-data (Add Input)"
os.environ["VKR_LAB"] = lab[0]
os.environ["VKR_RESULTS"] = R = "/kaggle/working/results"
os.environ["PYTHONWARNINGS"] = "ignore"
os.makedirs(f"{R}/preds", exist_ok=True)
print("Данные:", lab[0], "| граф:", os.path.exists(os.path.join(os.path.dirname(lab[0]), "graph")),
      "| events.csv:", os.path.exists(os.path.join(lab[0], "events.csv")))
print("Подключено в /kaggle/input:")
for p in sorted(glob.glob("/kaggle/input/*/*")):
    print("  ", p)

## 1. Прогнозы базовых моделей на val/test (из прошлых запусков)

In [ ]:
import pandas as pd
for p in sorted(glob.glob("/kaggle/input/**/results/preds/*.npz", recursive=True)):
    dst = f"{R}/preds/{os.path.basename(p)}"
    if not os.path.exists(dst):
        shutil.copy(p, dst)
logs = glob.glob("/kaggle/input/**/results/runs.csv", recursive=True)
if logs:
    pd.concat([pd.read_csv(p) for p in logs]).drop_duplicates().to_csv(f"{R}/runs.csv", index=False)
for p in glob.glob("/kaggle/input/**/results/fi_*.csv", recursive=True):
    shutil.copy(p, R)
need = [f"{m}__{s}.npz" for m in ("lgbm_final", "gnn_od_in", "chronos2") for s in ("val", "test")]
missing = [n for n in need if not os.path.exists(f"{R}/preds/{n}")]
assert not missing, f"Нет прогнозов {missing}: подключите вывод lgbm-final, gnn и chronos2"
!cd {REPO} && python src/baselines.py

In [ ]:
!cd {REPO} && python src/train_hybrid.py --leak-test

## 2. Базовые модели на периоде stack

In [ ]:
# Chronos-2 на GPU в фоне, пока LightGBM считается на процессоре
chronos_log = open(f"{R}/chronos_stack.log", "w")
chronos = None
if not os.path.exists(f"{R}/preds/chronos2__stack.npz"):
    chronos = subprocess.Popen(["python", "src/run_chronos.py", "--name", "chronos2", "--splits", "stack",
                                "--pred-stride", "2"], cwd=REPO, stdout=chronos_log, stderr=subprocess.STDOUT)
t = time.time()
if not os.path.exists(f"{R}/preds/lgbm_final__stack.npz"):
    !cd {REPO} && python src/train_lgbm.py --name lgbm_final --bias-correct --splits stack --pred-stride 2
print(f"lgbm_final (stack): {(time.time() - t) / 60:.0f} мин")

In [ ]:
t = time.time()
if chronos is not None:
    chronos.wait()
print(open(f"{R}/chronos_stack.log").read()[-1500:])
print(f"ожидание Chronos-2: {(time.time() - t) / 60:.0f} мин")
t = time.time()
if not os.path.exists(f"{R}/preds/gnn_od_in__stack.npz"):
    !cd {REPO} && python src/train_gnn.py --name gnn_od_in --graphs od_in --bias-correct --splits stack --pred-stride 2
print(f"gnn_od_in (stack): {(time.time() - t) / 60:.0f} мин")
print(sorted(os.path.basename(p) for p in glob.glob(f"{R}/preds/*__stack.npz")))

## 3. Гибрид

In [ ]:
t = time.time()
!cd {REPO} && python src/train_hybrid.py
print(f"гибрид: {(time.time() - t) / 60:.0f} мин")

## 4. Оценка

In [ ]:
!cd {REPO} && python src/evaluate.py --split val --ref norm_level

In [ ]:
!cd {REPO} && python src/evaluate.py --split test --ref norm_level

In [ ]:
PAIRS = " ".join([
    "hyb_full:lgbm_final", "hyb_blend:lgbm_final", "hyb_lgbm_only:lgbm_final",   # что даёт гибрид
    "hyb_nonbr:hyb_full", "hyb_no_resid:hyb_full",                                # соседи, недавние ошибки
    "hyb_no_chronos:hyb_full", "hyb_no_gnn:hyb_full",                             # вклад каждой базовой модели
])
!cd {REPO} && python src/report.py --split val --pairs {PAIRS}

In [ ]:
!cd {REPO} && python src/report.py --split test --pairs {PAIRS}

In [ ]:
pd.set_option("display.width", 200, "display.max_columns", 20)
H_ = ["горизонт h1", "горизонт h2-3", "горизонт h4-6", "горизонт h7-12", "горизонт h13-24"]
for split in ["val", "test"]:
    m = pd.read_csv(f"{R}/metrics_{split}.csv")
    C = m[(m.subset == "все точки") & (m.model.str.startswith("hyb_") |
          m.model.isin(["norm_level", "lgbm_final", "gnn_od_in", "chronos2"]))]
    t = C[C.slice == "всё"].set_index("model")[["wape", "bias"]].join(
        C[C.slice.isin(H_)].pivot_table(index="model", columns="slice", values="wape")[H_])
    print(f"\n{split}"); display(t.round(4).sort_values("wape"))

## 5. Архив

In [ ]:
import zipfile
with zipfile.ZipFile("/kaggle/working/results_csv.zip", "w", zipfile.ZIP_DEFLATED) as z:
    for p in glob.glob(f"{R}/*.csv") + glob.glob(f"{R}/*.log"):
        z.write(p, os.path.basename(p))
print(sorted(os.path.basename(p) for p in glob.glob(f"{R}/*.csv")))